# MolmoPoint — Image Pointing

MolmoPoint locates objects in images by **pointing** — returning precise pixel coordinates rather than bounding boxes. Given a natural language prompt like `"person"` or `"red car on the left"`, it finds every matching instance and returns one keypoint per object.

Each `fo.Keypoint` written to the dataset has:
- `label` — the object name from your prompt
- `points` — a single `[[x, y]]` coordinate pair, normalized to `[0, 1]`

Two ways to set the prompt:
- **Global prompt** — same objects searched across every sample
- **Per-sample prompt** — each sample is prompted with its own list of objects (read from a dataset field)

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/molmo_point",
    overwrite=True,
)

dataset = foz.load_zoo_dataset("quickstart")

model = foz.load_zoo_model("allenai/MolmoPoint-8B")


## Global prompt

Set `model.prompt` to a list of objects (or a comma-separated string) and every sample in the dataset is searched for all of them.

The model runs one generation pass per object per image, so keep the prompt focused to what you actually need.

In [ ]:
model.prompt = ["person", "animal", "drink", "food", "vehicle", "table", "chair"]

dataset.apply_model(
    model,
    label_field="points_from_global_prompt",
    batch_size=4,
    num_workers=2,
    skip_failures=False,
)

## Per-sample prompt

Pass `prompt_field` to `apply_model` and each sample is prompted with its own list of objects read from that field — no global prompt needed.

This is useful when your dataset already has ground-truth labels and you want to verify or augment them: each image is only asked about the objects that actually appear in it.

In [ ]:
# Derive unique object labels per sample from existing ground-truth detections
unique_objects = [
    list(set(labels))
    for labels in dataset.values("ground_truth.detections.label")
]
dataset.set_values("objs", unique_objects)

dataset.apply_model(
    model,
    prompt_field="objs",
    label_field="points_from_prompt_field",
    batch_size=4,
    num_workers=2,
    skip_failures=False,
)

## GUI / Screenshot model

`MolmoPoint-Img-8B` is fine-tuned specifically for UI elements in screenshots — buttons, input fields, menus, icons. Swap it in by changing the model name; everything else is identical.

Useful for:
- UI testing and automation workflows
- Annotating interactive elements in screenshots at scale
- Finding specific controls described in natural language

In [ ]:
gui_model = foz.load_zoo_model("allenai/MolmoPoint-Img-8B")
gui_model.prompt = ["submit button", "search bar", "navigation menu", "dropdown"]

# Point at your screenshot dataset
# dataset.apply_model(gui_model, label_field="ui_points", batch_size=4, num_workers=2)